# Kontrole rachunkowe zestawu problemów

Wyniki kontrolne dla P01–P10 oraz części liczbowych S02 i S04.
Sygnały kalibracyjne są w całości syntetyczne. Nie są to wyniki
notebooków rozdziałów ani pomiary pacjentów. Wskazówki interpretacyjne
znajdują się w samodzielnym tekście `wskazowki_do_problemow.md`.
Uruchomienie: `python scripts/run_problem_checks.py` z katalogu pakietu.
Wystarcza biblioteka standardowa Pythona.

In [1]:
from pathlib import Path
import math, cmath, json, platform, statistics
ROOT = next(p for p in [Path.cwd(), Path.cwd().parent]
            if (p/'scripts/run_problem_checks.py').is_file())
OUT = ROOT/'results'
OUT.mkdir(exist_ok=True)
RESULTS = {}
CHECKS = []
def check(name, condition):
    if not bool(condition):
        raise AssertionError(name)
    CHECKS.append(name)
    print('OK:', name)
def close(a,b,tol=1e-10):
    return math.isclose(a,b,rel_tol=tol,abs_tol=tol)
ENV = {'python':platform.python_version(), 'dependencies':'stdlib only',
       'execution':'ordinary Python cells in a fresh process'}

## P01–P03: oś, alias i skala DFT

In [2]:
physical_ms = 310-90+20
counters = [253,254,1,2]
steps = [(b-a)%256 for a,b in zip(counters,counters[1:])]
missing = sum(step-1 for step in steps)
check('P01: różnica fizjologicznych znaczników',physical_ms==240)
check('P01: zawinięcie licznika i luka',steps==[1,3,1] and missing==2 and close(missing/250,.008))
RESULTS['P01'] = {'physical_ms':physical_ms,'counter_steps':steps,'missing_samples':missing,'axis_shrink_ms':1000*missing/250}
fs = 200.
alias = abs(((130+fs/2)%fs)-fs/2)
samples_a = [math.cos(2*math.pi*130*n/fs) for n in range(200)]
samples_b = [math.cos(2*math.pi*70*n/fs) for n in range(200)]
delta_mv = 4/2**12
tolerance_mv = delta_mv/2
attenuation_db = 20*math.log10(.2/tolerance_mv)
check('P02: identyczne próbki 130 i 70 Hz',alias==70 and max(abs(a-b) for a,b in zip(samples_a,samples_b))<1e-11)
check('P02: kwantyzacja i budżet',close(delta_mv*1000,.9765625) and close(attenuation_db,52.24719895935549,1e-9))
RESULTS['P02'] = {'alias_hz':alias,'lsb_uV':1000*delta_mv,'tolerance_uV':1000*tolerance_mv,'attenuation_db':attenuation_db}
N,fs,A,f0 = 1500,500.,1.2,40.
x = [A*math.cos(2*math.pi*f0*n/fs) for n in range(N)]
grids = []
for nfft in [N,6000]:
    k = round(f0*nfft/fs)
    coefficient = sum(v*cmath.exp(-2j*math.pi*k*n/nfft) for n,v in enumerate(x))
    grids.append({'nfft':nfft,'df_hz':fs/nfft,'k':k,'module':abs(coefficient),'amplitude_mV':2*abs(coefficient)/N})
check('P03: czas i siatki',N/fs==3 and [r['k'] for r in grids]==[120,480] and close(grids[1]['df_hz'],1/12))
check('P03: bezpośrednia suma DFT i normalizacja',all(close(r['module'],900,1e-8) and close(r['amplitude_mV'],1.2,1e-8) for r in grids))
RESULTS['P03'] = {'duration_s':N/fs,'variants':grids,'wrong_amplitude_mV':2*grids[1]['module']/6000}

OK: P01: różnica fizjologicznych znaczników
OK: P01: zawinięcie licznika i luka
OK: P02: identyczne próbki 130 i 70 Hz
OK: P02: kwantyzacja i budżet
OK: P03: czas i siatki
OK: P03: bezpośrednia suma DFT i normalizacja


## P04–P07: momenty, czas i reprezentacja lokalna

In [3]:
values = [1.,7.,1.,7.]
mean = statistics.mean(values)
variance = statistics.mean((v-mean)**2 for v in values)
m2 = statistics.mean(v*v for v in values)
K = {str(overlap):len(range(0,6000-1000+1,1000-overlap)) for overlap in [500,750]}
check('P04: momenty ze skończonego ciągu',mean==4 and variance==9 and m2==25 and math.sqrt(m2)==5)
check('P04: pełne segmenty z enumeracji',K=={'500':11,'750':21})
# Bez niezależności i wspólnej wariancji ta reguła nie stanowi przewidywania dla epok pomiarowych.
check('P04: idealne SD średniej',10/math.sqrt(25)==2)
RESULTS['P04'] = {'mean_uV':mean,'variance_uV2':variance,'mean_square_uV2':m2,'rms_uV':math.sqrt(m2),'segments':K,'averaged_noise_sd_uV':2.}
L,fs_in,fs_out = 151,500.,200.
delay_samples = (L-1)/2
index_out = round(2.348*fs_out)
time_out = index_out/fs_out
check('P05: rząd i opóźnienie',L-1==150 and delay_samples==75 and close(delay_samples/fs_in,.15))
check('P05: stosunek siatek i błąd znacznika',fs_in*2/5==fs_out and index_out==470 and close(1000*(time_out-2.348),2))
RESULTS['P05'] = {'order':150,'delay_samples':delay_samples,'delay_s':delay_samples/fs_in,'U':2,'D':5,'event_index_out':index_out,'event_time_out_s':time_out,'rounding_error_ms':1000*(time_out-2.348)}
phase = lambda t:2*math.pi*(18*t+3*t*t)
dt=1e-5
frequency = (phase(2+dt)-phase(2-dt))/(2*dt)/(2*math.pi)
lags = [math.ceil(60*80/180), math.floor(60*80/80)]
check('P06: pochodna fazy w Hz',close(frequency,30.,1e-8))
check('P06: granice lagu i rytm',lags==[27,60] and 60*80/40==120 and 60/.5==120)
RESULTS['P06'] = {'instantaneous_hz':frequency,'packet_rate_per_min':120.,'lag_bounds':lags,'target_lag':40}
frames = list(range(0,1440-120+1,30))
scales = [240/12,240/48]
edge_s = 8*scales[0]/240
check('P07: ramki i siatki',len(frames)==45 and 120/240==.5 and 30/240==.125 and 240/960==.25 and 120-30==90)
check('P07: skale i promień ostrzegawczy',scales==[20.,5.] and close(edge_s,2/3))
RESULTS['P07'] = {'frames':len(frames),'window_s':.5,'hop_s':.125,'overlap_samples':90,'df_hz':.25,'scales':scales,'edge_radius_s':edge_s}

OK: P04: momenty ze skończonego ciągu
OK: P04: pełne segmenty z enumeracji
OK: P04: idealne SD średniej
OK: P05: rząd i opóźnienie
OK: P05: stosunek siatek i błąd znacznika
OK: P06: pochodna fazy w Hz
OK: P06: granice lagu i rytm
OK: P07: ramki i siatki
OK: P07: skale i promień ostrzegawczy


## P08: indeks pełnego splotu i tożsamość skalowania LMS

Kalibracja używa wzorca o znanym początku. Próba LMS porównuje dwa
algebraicznie równoważne skalowania referencji, z zerowymi wagami.

In [4]:
length,start = 120,680
pattern = [math.exp(-.5*((n-60)/12)**2)*math.cos(2*math.pi*38*n/500) for n in range(length)]
calibration = [0.]*1000
calibration[start:start+length] = pattern
h = pattern[::-1]
full = [0.]*(len(calibration)+len(h)-1)
for n,v in enumerate(calibration):
    if v:
        for j,w in enumerate(h):full[n+j]+=v*w
kmax = max(range(len(full)),key=full.__getitem__)
estimated_start = kmax-(length-1)
center = estimated_start+length//2
check('P08: kalibracja początku i środka splotu',kmax==799 and estimated_start==680 and center==740 and center/500==1.48)
u = [math.sin(.31*n)+.3*math.cos(.13*n) for n in range(256)]
d = [math.sin(.07*n)+.4*v for n,v in enumerate(u)]
def lms_reference(d,u,mu,length=4):
    weights=[0.]*length
    result=[]
    history=[]
    for n in range(length-1,len(u)):
        v=[u[n-j] for j in range(length)]
        error=d[n]-sum(a*b for a,b in zip(weights,v))
        weights=[a+mu*error*b for a,b in zip(weights,v)]
        result.append(error)
        history.append(weights)
    return result,history
baseline,wb = lms_reference(d,u,.001)
scaled,ws = lms_reference(d,[4*v for v in u],.001/16)
check('P08: przebieg LMS zachowuje skalowanie',max(abs(a-b) for a,b in zip(baseline,scaled))<1e-12 and max(abs(a/4-b) for row_a,row_b in zip(wb,ws) for a,b in zip(row_a,row_b))<1e-12)
RESULTS['P08'] = {'convolution_max_index':kmax,'event_start_index':estimated_start,'event_center_index':center,'event_center_s':center/500,'mu_scaled':.001/16,'weight_factor':.25,'max_output_scale_difference':max(abs(a-b) for a,b in zip(baseline,scaled))}

OK: P08: kalibracja początku i środka splotu
OK: P08: przebieg LMS zachowuje skalowanie


## P09–P10: mianowniki i agregacja

In [5]:
reference,detected = [100,300,500],[108,308,508,700]
fs,tolerance_ms=200.,50.
tolerance=round(tolerance_ms*fs/1000)
eligible=[(i,j) for i,r in enumerate(reference) for j,q in enumerate(detected) if abs(q-r)<=tolerance]
# W tym zadaniu pary są rozłączne; nie implementujemy ogólnego algorytmu optymalnego parowania.
assert len({i for i,j in eligible})==len(eligible)==len({j for i,j in eligible})
TP,FP,FN=len(eligible),len(detected)-len(eligible),len(reference)-len(eligible)
mae_ms=statistics.mean(abs(detected[j]-reference[i])*1000/fs for i,j in eligible)
Se,PPV,F1=TP/(TP+FN),TP/(TP+FP),2*TP/(2*TP+FP+FN)
rr_reference=[(b-a)/fs for a,b in zip(reference,reference[1:])]
rr_detected=[(b-a)/fs for a,b in zip(detected,detected[1:])]
check('P09: pary, liczniki, trafność i czas',tolerance==10 and (TP,FP,FN)==(3,1,0) and Se==1 and PPV==.75 and close(F1,6/7) and mae_ms==40)
check('P09: zgodna mediana RR nie usuwa błędów',statistics.median(rr_reference)==statistics.median(rr_detected)==1)
RESULTS['P09']={'tolerance_samples':tolerance,'pairs_0based':eligible,'TP':TP,'FP':FP,'FN':FN,'Se':Se,'PPV':PPV,'F1':F1,'MAE_ms':mae_ms,'RR_reference_s':rr_reference,'RR_detected_s':rr_detected,'HR_both_per_min':60.}
coverage_a,coverage_b=16/20,8/20
mae_a,mae_b=statistics.mean([2.]*16),statistics.mean([1.]*8)
check('P10: pokrycie i błąd na właściwych zbiorach',coverage_a==.8 and coverage_b==.4 and mae_a==2 and mae_b==1)
records=[(90,10,10),(9,0,1)]
total=[sum(row[j] for row in records) for j in range(3)]
tp,fp,fn=total
micro=2*tp/(2*tp+fp+fn)
macro=statistics.mean(2*tp/(2*tp+fp+fn) for tp,fp,fn in records)
check('P10: micro i macro mają różne wagi',close(micro,198/219) and close(macro,(.9+18/19)/2) and micro<macro)
RESULTS['P10']={'coverage_A':coverage_a,'coverage_B':coverage_b,'MAE_A_per_min':mae_a,'MAE_B_per_min':mae_b,'empty_comparison_error':None,'F1_micro':micro,'F1_macro':macro}

OK: P09: pary, liczniki, trafność i czas
OK: P09: zgodna mediana RR nie usuwa błędów
OK: P10: pokrycie i błąd na właściwych zbiorach
OK: P10: micro i macro mają różne wagi


## S02 i S04: liczby w zadaniach przekrojowych

In [6]:
N=32*256
welch_starts=list(range(0,N-512+1,256))
stft_starts=list(range(0,N-512+1,64))
check('S02: segmenty, ramki i czas',len(welch_starts)==31 and len(stft_starts)==121 and 512/256==2 and 256/2048==.125)
RESULTS['S02']={'N':N,'welch_segments':len(welch_starts),'stft_frames':len(stft_starts),'window_s':2.,'df_hz':.125}
fs=8000.
delay_s=680/fs
periods=[nfft/fs for nfft in [16384,32768]]
check('S04: quefrencja i okres dopełnienia',close(delay_s,.085) and close(1/fs,.000125) and periods==[2.048,4.096])
RESULTS['S04']={'delay_s':delay_s,'quefrency_step_s':1/fs,'grid_periods_s':periods}
RESULTS['audit']={'passed_checks':len(CHECKS),'checks':CHECKS}
(OUT/'problem_checks.json').write_text(json.dumps(RESULTS,ensure_ascii=False,indent=2)+'\n',encoding='utf-8')
(OUT/'problem_environment.json').write_text(json.dumps(ENV,ensure_ascii=False,indent=2)+'\n',encoding='utf-8')
print('Kontrole rachunkowe:',len(CHECKS))

OK: S02: segmenty, ramki i czas
OK: S04: quefrencja i okres dopełnienia
Kontrole rachunkowe: 23
